In [ ]:
pip install PyPDF2


In [ ]:
###SST anos individuais

import requests
from tqdm import tqdm
import os
import time
import PyPDF2

# Link base para os arquivos
base_url = 'https://www.star.nesdis.noaa.gov/pub/socd/mecb/crw/data/5km/v3.1_op/nc/v1.0/daily/sst/2025/'

# Caminho do diretório onde está o arquivo PDF
pdf_dir = r'C:\Users\rbfra\OneDrive\########CEBIMAR\####PROJETOS\#####Coral trade offs\remote sensing\\'

# Nome do arquivo PDF
pdf_file = 'https___www.star.nesdis.noaa.gov_pub_socd_mecb_crw_data_5km_v3.1_op_nc_v1.0_daily_sst_2025_ - NOAA _ NESDIS _ STAR File Share.pdf'

# Solicita o diretório de saída para as imagens
output_dir = input("Digite o caminho do diretório de saída para salvar as imagens: ")

# Verifica se o diretório existe; se não, cria-o
os.makedirs(output_dir, exist_ok=True)

# Função para limpar e validar nomes de arquivos
def clean_filename(filename):
    filename = filename.strip()
    if filename.endswith('.nc'):
        return filename
    return None

# Função para extrair nomes de arquivos de um arquivo PDF
def extract_filenames_from_pdf(pdf_path):
    filenames = []
    with open(pdf_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        for page in reader.pages:
            text = page.extract_text()
            # Extrai e filtra apenas nomes que terminam com .nc
            raw_filenames = [word for word in text.split() if word.endswith('.nc')]
            filenames.extend([clean_filename(name) for name in raw_filenames if clean_filename(name) is not None])
    return filenames

# Função para baixar uma imagem com verificação de progresso
def download_image(url, output_path):
    try:
        response = requests.get(url, stream=True, timeout=10)
        
        if response.status_code == 200:
            total_size = int(response.headers.get('content-length', 0))
            if total_size == 0:
                print(f"URL inválida ou arquivo vazio: {url}")
                return
            
            with open(output_path, 'wb') as file, tqdm(
                desc=f"Baixando {os.path.basename(output_path)}",
                total=total_size,
                unit='B',
                unit_scale=True,
                unit_divisor=1024,
                leave=True
            ) as pbar:
                for chunk in response.iter_content(1024):
                    file.write(chunk)
                    pbar.update(len(chunk))

            print(f"Download concluído: {os.path.basename(output_path)}")
        else:
            print(f"Falha ao baixar (Status {response.status_code}): {url}")
    
    except requests.exceptions.Timeout:
        print(f"Tempo de conexão esgotado para {url}")
    except requests.exceptions.RequestException as e:
        print(f"Erro ao baixar {url}: {e}")

# Caminho completo do arquivo PDF
pdf_path = os.path.join(pdf_dir, pdf_file)

# Extrai nomes de arquivos do PDF
print(f"\nProcessando {pdf_file}...")
filenames = extract_filenames_from_pdf(pdf_path)

if not filenames:
    print(f"Nenhum arquivo .nc encontrado no arquivo {pdf_file}")
else:
    # Loop para construir URLs e baixar os arquivos
    for i, filename in enumerate(filenames, start=1):
        url = base_url + filename
        output_path = os.path.join(output_dir, filename)

        print(f"\n[{i}/{len(filenames)}] Baixando: {url}")
        start_time = time.time()
        download_image(url, output_path)
        end_time = time.time()

        print(f"Tempo para {os.path.basename(output_path)}: {end_time - start_time:.2f} segundos")

print("\nDownload concluído para todos os arquivos .nc no PDF!")

In [ ]:
###DHW all links on folder

import requests
from tqdm import tqdm
import os
import time
import PyPDF2
import re

# Caminho do diretório onde estão os PDFs com os links dos arquivos .nc
pdf_dir = r'C:\Users\rbfra\OneDrive\########CEBIMAR\####PROJETOS\#####Coral trade offs\remote sensing\@DHW_Links'

# Solicita o diretório de saída para salvar os arquivos .nc
output_dir = input("Digite o caminho do diretório de saída para salvar os arquivos .nc: ")
os.makedirs(output_dir, exist_ok=True)

def clean_filename_revised(filename_str):
    """
    Limpa e valida uma string de nome de arquivo.
    - Remove espaços em branco no início/fim da string completa.
    - Substitui caracteres inválidos do sistema de arquivos no nome base por sublinhados.
    - Garante que o nome do arquivo não esteja vazio após a limpeza (excluindo a extensão).
    - Retorna o nome limpo se ele originalmente terminava com .nc, caso contrário retorna None.
    """
    # Remove espaços em branco da string inteira primeiro
    cleaned_name = filename_str.strip()

    # Verifica se o nome realmente termina com .nc; crucial para não processar e.g. .nc.md5
    if not cleaned_name.endswith('.nc'):
        return None

    # Separa o nome base e a extensão para não danificar a extensão
    base_name, ext = os.path.splitext(cleaned_name) # ext será '.nc'

    # Sanitiza o nome base
    # Substitui caracteres inválidos (ex: < > : " / \ | ? *) e caracteres de controle por um sublinhado
    base_name = re.sub(r'[<>:"/\\|?*\x00-\x1F]', '_', base_name)
    
    # Remove pontos ou sublinhados no início/fim do nome base que podem ter sido criados pela substituição
    base_name = base_name.strip('._')
    
    # Substitui múltiplos sublinhados (possivelmente de múltiplas substituições) por um único
    base_name = re.sub(r'_+', '_', base_name)

    # Garante que o nome base não esteja vazio após a limpeza
    if not base_name:
        # Isso pode acontecer se o nome original fosse algo como ":.:.nc"
        return None

    return base_name + ext # ext é '.nc'

def extract_nc_filenames_from_pdf(pdf_path):
    """
    Extrai os nomes dos arquivos .nc de um arquivo PDF.
    Percorre todas as páginas e separa as palavras que terminem com .nc.
    Aplica uma limpeza robusta aos nomes de arquivo extraídos.
    """
    filenames = []
    try:
        with open(pdf_path, 'rb') as file:
            reader = PyPDF2.PdfReader(file)
            for page in reader.pages:
                text = page.extract_text()
                if text:
                    # Separa as palavras e filtra aquelas que, após remover espaços extras, terminam com .nc
                    potential_filenames_candidates = text.split()
                    
                    for name_candidate_raw in potential_filenames_candidates:
                        # name_candidate_raw é uma palavra do texto do PDF, ex: "  arquivo:nome.nc  "
                        # clean_filename_revised lidará com a remoção de espaços e sanitização.
                        # A verificação .endswith('.nc') dentro de clean_filename_revised é a definitiva.
                        cleaned_name = clean_filename_revised(name_candidate_raw) 
                        if cleaned_name:
                            filenames.append(cleaned_name)
    except Exception as e:
        print(f"Erro ao processar {pdf_path}: {e}")
    return filenames

def download_file(url, output_path):
    """
    Realiza o download do arquivo .nc a partir da URL e salva no caminho especificado,
    exibindo uma barra de progresso. (Versão mais robusta)
    """
    try:
        print(f"Tentando baixar de: {url}")
        print(f"Salvando em: {output_path}")
        response = requests.get(url, stream=True, timeout=30) # Timeout aumentado para 30s
        
        if response.status_code == 200:
            total_size = int(response.headers.get('content-length', 0))
            if total_size == 0:
                # Considerar se um arquivo de tamanho 0 é um erro ou um caso válido.
                print(f"Aviso: Cabeçalho 'content-length' é 0 ou ausente para {url}. O arquivo pode estar vazio ou o servidor não o forneceu.")

            with open(output_path, 'wb') as file, tqdm(
                desc=f"Baixando {os.path.basename(output_path)}",
                total=total_size,
                unit='B',
                unit_scale=True,
                unit_divisor=1024,
                leave=True
            ) as pbar:
                for chunk in response.iter_content(1024):
                    if chunk: # Garante que o chunk não está vazio
                        file.write(chunk)
                        pbar.update(len(chunk))
            
            if total_size > 0 and os.path.exists(output_path) and os.path.getsize(output_path) < total_size:
                print(f"Aviso: Download de {os.path.basename(output_path)} pode estar incompleto. Esperado: {total_size}, Baixado: {os.path.getsize(output_path)}")
            elif os.path.exists(output_path) and os.path.getsize(output_path) == 0 and total_size == 0 :
                 print(f"Aviso: {os.path.basename(output_path)} foi baixado e tem 0 bytes (content-length também era 0 ou ausente).")
            elif not os.path.exists(output_path) and response.status_code == 200:
                print(f"Erro: O arquivo {os.path.basename(output_path)} não foi salvo apesar do status 200.")
            else:
                print(f"Download concluído: {os.path.basename(output_path)}")

        else:
            print(f"Falha ao baixar (Status {response.status_code}): {url}")
    
    except requests.exceptions.Timeout:
        print(f"Tempo de conexão esgotado para {url}")
    except requests.exceptions.RequestException as e:
        print(f"Erro ao baixar {url}: {e}")
    except OSError as e: 
        print(f"Erro de sistema ao escrever o arquivo {output_path}: {e}")
    except Exception as e: 
        print(f"Um erro inesperado ocorreu durante o download de {url} para {output_path}: {e}")


def extract_year_from_pdf(pdf_filename):
    """
    Extrai o ano do nome do arquivo PDF.
    """
    match = re.search(r'\d{4}', pdf_filename)
    if match:
        return match.group(0)
    return None

# Lista todos os arquivos PDF na pasta especificada
pdf_files = [f for f in os.listdir(pdf_dir) if f.lower().endswith('.pdf')]

if not pdf_files:
    print("Nenhum arquivo PDF encontrado no diretório especificado.")
else:
    for pdf_file in pdf_files:
        pdf_path = os.path.join(pdf_dir, pdf_file)
        print(f"\nProcessando o arquivo PDF: {pdf_file}...")
        nc_filenames = extract_nc_filenames_from_pdf(pdf_path)
        
        if not nc_filenames:
            print(f"Nenhum arquivo .nc válido encontrado no PDF {pdf_file}")
        else:
            year = extract_year_from_pdf(pdf_file)
            if not year:
                print(f"Não foi possível extrair o ano do nome do arquivo PDF {pdf_file}. Pulando este PDF.")
                continue
            
            # Base URL específica para DHW
            base_url = f'https://www.star.nesdis.noaa.gov/pub/socd/mecb/crw/data/5km/v3.1_op/nc/v1.0/daily/dhw/{year}/'
            
            print(f"Encontrados {len(nc_filenames)} arquivos .nc em {pdf_file} para o ano {year}.")
            for i, nc_filename in enumerate(nc_filenames, start=1):
                # nc_filename já foi limpo por clean_filename_revised
                url = base_url + nc_filename 
                output_file = os.path.join(output_dir, nc_filename)
                
                print(f"\n[{i}/{len(nc_filenames)}] Preparando para baixar: {nc_filename}")
                start_time = time.time()
                download_file(url, output_file)
                elapsed = time.time() - start_time
                print(f"Tempo para operação de {nc_filename}: {elapsed:.2f} segundos")
                # Adiciona um pequeno delay para não sobrecarregar o servidor
                time.sleep(0.5) 

print("\nProcessamento de todos os PDFs concluído!")

In [ ]:
###SST all links on folder

import requests
from tqdm import tqdm
import os
import time
import PyPDF2
import re

# Caminho do diretório onde estão os PDFs com os links dos arquivos .nc
pdf_dir = r'C:\Users\rbfra\OneDrive\########CEBIMAR\####PROJETOS\#####Coral trade offs\remote sensing\@SST_Links'

# Solicita o diretório de saída para salvar os arquivos .nc
output_dir = input("Digite o caminho do diretório de saída para salvar os arquivos .nc: ")
os.makedirs(output_dir, exist_ok=True)

def clean_filename_revised(filename_str):
    """
    Limpa e valida uma string de nome de arquivo.
    - Remove espaços em branco no início/fim da string completa.
    - Substitui caracteres inválidos do sistema de arquivos no nome base por sublinhados.
    - Garante que o nome do arquivo não esteja vazio após a limpeza (excluindo a extensão).
    - Retorna o nome limpo se ele originalmente terminava com .nc, caso contrário retorna None.
    """
    # Remove espaços em branco da string inteira primeiro
    cleaned_name = filename_str.strip()

    # Verifica se o nome realmente termina com .nc; crucial para não processar e.g. .nc.md5
    if not cleaned_name.endswith('.nc'):
        return None

    # Separa o nome base e a extensão para não danificar a extensão
    base_name, ext = os.path.splitext(cleaned_name) # ext será '.nc'

    # Sanitiza o nome base
    # Substitui caracteres inválidos (ex: < > : " / \ | ? *) e caracteres de controle por um sublinhado
    base_name = re.sub(r'[<>:"/\\|?*\x00-\x1F]', '_', base_name)
    
    # Remove pontos ou sublinhados no início/fim do nome base que podem ter sido criados pela substituição
    base_name = base_name.strip('._')
    
    # Substitui múltiplos sublinhados (possivelmente de múltiplas substituições) por um único
    base_name = re.sub(r'_+', '_', base_name)

    # Garante que o nome base não esteja vazio após a limpeza
    if not base_name:
        # Isso pode acontecer se o nome original fosse algo como ":.:.nc"
        return None

    return base_name + ext # ext é '.nc'

def extract_nc_filenames_from_pdf(pdf_path):
    """
    Extrai os nomes dos arquivos .nc de um arquivo PDF.
    Percorre todas as páginas e separa as palavras que terminam com .nc.
    Aplica uma limpeza robusta aos nomes de arquivo extraídos.
    """
    filenames = []
    try:
        with open(pdf_path, 'rb') as file:
            reader = PyPDF2.PdfReader(file)
            for page in reader.pages:
                text = page.extract_text()
                if text:
                    # Separa as palavras e filtra aquelas que, após remover espaços extras, terminam com .nc
                    potential_filenames_candidates = text.split()
                    
                    for name_candidate_raw in potential_filenames_candidates:
                        # name_candidate_raw é uma palavra do texto do PDF, ex: "  arquivo:nome.nc  "
                        # clean_filename_revised lidará com a remoção de espaços e sanitização.
                        # A verificação .endswith('.nc') dentro de clean_filename_revised é a definitiva.
                        cleaned_name = clean_filename_revised(name_candidate_raw) 
                        if cleaned_name:
                            filenames.append(cleaned_name)
    except Exception as e:
        print(f"Erro ao processar {pdf_path}: {e}")
    return filenames

def download_file(url, output_path):
    """
    Realiza o download do arquivo .nc a partir da URL e salva no caminho especificado,
    exibindo uma barra de progresso.
    """
    try:
        print(f"Tentando baixar de: {url}")
        print(f"Salvando em: {output_path}")
        response = requests.get(url, stream=True, timeout=30) # Timeout aumentado para 30s
        
        if response.status_code == 200:
            total_size = int(response.headers.get('content-length', 0))
            if total_size == 0:
                # Considerar se um arquivo de tamanho 0 é um erro ou um caso válido.
                # Para arquivos .nc, geralmente não são 0 bytes se válidos.
                print(f"Aviso: Cabeçalho 'content-length' é 0 ou ausente para {url}. O arquivo pode estar vazio ou o servidor não o forneceu.")
                # Se um arquivo de 0 bytes for baixado, ele será criado. Se isso não for desejado, descomente o return abaixo.
                # print(f"URL inválida ou arquivo vazio (content-length 0): {url}")
                # return

            with open(output_path, 'wb') as file, tqdm(
                desc=f"Baixando {os.path.basename(output_path)}",
                total=total_size,
                unit='B',
                unit_scale=True,
                unit_divisor=1024,
                leave=True
            ) as pbar:
                for chunk in response.iter_content(1024):
                    if chunk: # Garante que o chunk não está vazio
                        file.write(chunk)
                        pbar.update(len(chunk))
            
            # Verifica o tamanho do arquivo após o download se total_size era > 0
            if total_size > 0 and os.path.getsize(output_path) < total_size:
                print(f"Aviso: Download de {os.path.basename(output_path)} pode estar incompleto. Esperado: {total_size}, Baixado: {os.path.getsize(output_path)}")
            elif os.path.getsize(output_path) == 0 and total_size == 0 :
                 print(f"Aviso: {os.path.basename(output_path)} foi baixado e tem 0 bytes (content-length também era 0 ou ausente).")
            else:
                print(f"Download concluído: {os.path.basename(output_path)}")

        else:
            print(f"Falha ao baixar (Status {response.status_code}): {url}")
    
    except requests.exceptions.Timeout:
        print(f"Tempo de conexão esgotado para {url}")
    except requests.exceptions.RequestException as e:
        print(f"Erro ao baixar {url}: {e}")
    except OSError as e: # Captura explicitamente OSError que pode ocorrer em file.write
        print(f"Erro de sistema ao escrever o arquivo {output_path}: {e}")
    except Exception as e: # Captura genérica para outros erros inesperados
        print(f"Um erro inesperado ocorreu durante o download de {url} para {output_path}: {e}")


def extract_year_from_pdf(pdf_filename):
    """
    Extrai o ano do nome do arquivo PDF.
    """
    match = re.search(r'\d{4}', pdf_filename)
    if match:
        return match.group(0)
    return None

# Lista todos os arquivos PDF na pasta especificada
pdf_files = [f for f in os.listdir(pdf_dir) if f.lower().endswith('.pdf')]

if not pdf_files:
    print("Nenhum arquivo PDF encontrado no diretório especificado.")
else:
    for pdf_file in pdf_files:
        pdf_path = os.path.join(pdf_dir, pdf_file)
        print(f"\nProcessando o arquivo PDF: {pdf_file}...")
        nc_filenames = extract_nc_filenames_from_pdf(pdf_path)
        
        if not nc_filenames:
            print(f"Nenhum arquivo .nc válido encontrado no PDF {pdf_file}")
        else:
            year = extract_year_from_pdf(pdf_file)
            if not year:
                print(f"Não foi possível extrair o ano do nome do arquivo PDF {pdf_file}. Pulando este PDF.")
                continue
            
            # Base URL - confirme se o ano deve estar no final ou em outro lugar se a estrutura mudar
            base_url = f'https://www.star.nesdis.noaa.gov/pub/socd/mecb/crw/data/5km/v3.1_op/nc/v1.0/daily/sst/{year}/'
            
            print(f"Encontrados {len(nc_filenames)} arquivos .nc em {pdf_file} para o ano {year}.")
            for i, nc_filename in enumerate(nc_filenames, start=1):
                # nc_filename já foi limpo por clean_filename_revised
                url = base_url + nc_filename 
                output_file = os.path.join(output_dir, nc_filename)
                
                print(f"\n[{i}/{len(nc_filenames)}] Preparando para baixar: {nc_filename}")
                start_time = time.time()
                download_file(url, output_file)
                elapsed = time.time() - start_time
                print(f"Tempo para operação de {nc_filename}: {elapsed:.2f} segundos")
                # Adiciona um pequeno delay para não sobrecarregar o servidor
                time.sleep(0.5) 

print("\nProcessamento de todos os PDFs concluído!")

In [ ]:
pip install pypdf2
